In [ ]:
#!git clone https://github.com/Jun1801/CoDraft-Bench.git

In [ ]:
%pip install -r requirements.txt

In [ ]:
%cd "YOUR_WORKING_DIR"
CD_PATH = "YOUR WORKING DIR"

In [ ]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if CD_PATH not in sys.path:
    sys.path.append(CD_PATH)

In [ ]:
from copy import deepcopy
import numpy as np
import pandas as pd
import torch

from config import *
from preprocess import *
from model import *
from model.models import *
from utils import *

In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
%cd ..

In [ ]:
INPUT_ROOT = "dataset"
WORK_DIR = "working"
MODEL_TYPE = "ml_base"
MODEL_NAME = "microsoft/deberta-v3-base"


In [ ]:
tokenizer = get_tokenizer(MODEL_NAME)

In [ ]:
data_manager = DataManager(
    input_root=INPUT_ROOT, 
    work_dir=WORK_DIR, 
    config_data=CONFIG_DATA,
    seed_worker=seed_worker,
    tokenizer=tokenizer,
    data_generator=data_generator,
    random_seed = RANDOM_SEED,
    rebalance=False,
    train_file="train.csv",
    val_file="val.csv",
    test_file="test.csv")

In [ ]:
train_df, val_df, test_df = data_manager.get_data()
X_train, y_train, X_val, y_val, X_test, y_test = data_manager.get_ml_data()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_weights = compute_class_weight(train_df['label_score'])


In [ ]:
model, vectorizer = train_svm(X_train, y_train)

In [ ]:
model, vectorizer = train_xgboost(X_train, y_train, X_val, y_val, class_weights)
#model, vectorizer = train_svm(X_train, y_train)


In [ ]:
test_preds, test_true = get_preds_ml(model, X_test, y_test)


In [ ]:
result_df = pd.DataFrame({
    "label": test_true,
    "pred": test_preds,
})

In [ ]:
result_df.to_csv("result_df.csv", index=False)

In [ ]:
get_stats(result_df)